# GenIDS-CIC18: consolidation of NFStream flow files

This notebook consolidates the ten CIC-IDS2018 CSV files generated by the extraction and daily-labeling notebooks. It validates their schemas, maps the original attack labels to the GenIDS multiclass representation, removes non-model and duplicate columns/rows, and exports the complete GenIDS-CIC18 dataset. Only the paths in the configuration cell should be changed.

## 1. Configuration

In [ ]:
from pathlib import Path

INPUT_DIR = Path("/path/to/output/nfstream_daily_csv")
OUTPUT_DIR = Path("/path/to/output")
OUTPUT_FILE = OUTPUT_DIR / "GenIDS-CIC18.csv"
INPUT_FILES = {
    "Experiment 01": "01_cic2018_tuesday_nfstream_20_02_18.csv",
    "Experiment 02": "02_cic2018_wednesday_nfstream_14_02_18.csv",
    "Experiment 03": "03_cic2018_wednesday_nfstream_ddos_21_02_18.csv",
    "Experiment 04": "04_cic2018_wednesday_nfstream_28_02_18.csv",
    "Experiment 05": "05_cic2018_thursday_nfstream_dos_15_02_18.csv",
    "Experiment 06": "06_cic2018_thursday_nfstream_webattack_22_02_18.csv",
    "Experiment 07": "07_cic2018_thursday_nfstream_infiltration_01_03_18.csv",
    "Experiment 08": "08_cic2018_friday_nfstream_dos_16_02_18.csv",
    "Experiment 09": "09_cic2018_friday_nfstream_webattack_23_02_18.csv",
    "Experiment 10": "10_cic2018_friday_nfstream_botnet_02_03_18.csv",
}
RANDOM_STATE = 42
MAX_DDOS_FLOWS = 1_100_000
REQUIRED_COLUMNS = {"label", "multiclass"}

## 2. Imports and helper functions

In [ ]:
import pandas as pd

def validate_files():
    missing=[INPUT_DIR/name for name in INPUT_FILES.values() if not (INPUT_DIR/name).is_file()]
    if missing: raise FileNotFoundError("Missing daily CSV files:\n"+"\n".join(f"- {p}" for p in missing))

def load_files():
    frames=[]; records=[]; reference_columns=None
    for experiment,name in INPUT_FILES.items():
        frame=pd.read_csv(INPUT_DIR/name,low_memory=False)
        missing=REQUIRED_COLUMNS.difference(frame.columns)
        if missing: raise ValueError(f"{name} is missing columns: {sorted(missing)}")
        current=set(frame.columns)
        if reference_columns is None: reference_columns=current
        elif current!=reference_columns: raise ValueError(f"Schema mismatch in {name}.")
        frames.append(frame); records.append({"source":experiment,"file":name,"flows":len(frame)})
    return pd.concat(frames,ignore_index=True),pd.DataFrame(records)

def map_genids_class(value):
    value=str(value).strip().lower()
    if value=="benign": return "benign"
    if value.startswith("ddos_") or value.startswith("dos_") or value in {"ddos","dos"}: return "ddos"
    return "background"

def summarize(frame,column):
    return pd.DataFrame({"count":frame[column].value_counts(),"percentage":frame[column].value_counts(normalize=True).mul(100).round(2)})

## 3. Load and consolidate the ten flow files

In [ ]:
validate_files()
flows,source_summary=load_files()
display(source_summary)
print(f"Total loaded flows: {len(flows):,}")

## 4. Standardize labels and control the DDoS class

In [ ]:
flows["binary"]=flows["label"].map(lambda value: "benign" if str(value).lower()=="benign" else "malign")
flows["multiclass"]=flows["multiclass"].map(map_genids_class)
flows=flows.drop(columns=["label","id","expiration_id","Timestamp"],errors="ignore")

ddos=flows.loc[flows["multiclass"]=="ddos"]
other=flows.loc[flows["multiclass"]!="ddos"]
if len(ddos)>MAX_DDOS_FLOWS:
    ddos=ddos.sample(n=MAX_DDOS_FLOWS,random_state=RANDOM_STATE)
flows=pd.concat([other,ddos],ignore_index=True)

## 5. Remove duplicates and summarize the complete dataset

In [ ]:
rows_before=len(flows)
flows=flows.drop_duplicates().reset_index(drop=True)
labels=["binary","multiclass"]
features=[column for column in flows.columns if column not in labels]
genids_cic18=flows[features+labels]
print(f"Duplicate rows removed: {rows_before-len(genids_cic18):,}")
display(summarize(genids_cic18,"binary"))
display(summarize(genids_cic18,"multiclass"))

## 6. Export

In [ ]:
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
genids_cic18.to_csv(OUTPUT_FILE,index=False)
print(f"Dataset saved to: {OUTPUT_FILE.resolve()}")
print(f"Final shape: {genids_cic18.shape}")